In [ ]:
最適なペア順に信号線値を並べ替えたファイル（s~simular_output）をもとに、最適なペアの信号線を統合し、正解データを作成
※s~simular_outputファイルは、s~outputファイルの行と列が入れ替わっていることに注意⇒詳しくはokinaka.txtを参照

In [ ]:
# 学習済みのANNを用いて学習を行うプログラム
# diagnosis.ipynbを改良したもの、信号線数が奇数の場合を考慮　⇒　作成途
# 以下のプログラムが記述されたところから途中になっている
    # # 信号線ペアのIDをリストに格納
    # print("line_id:" + str(integration_num * len(lines)))
    # line_id = [0 for _ in range(signal_num)]  # 故障信号線のIDを格納するリスト
    # for i in range(len(lines) - (signal_num%integration_num)):    # signal_num%integration_numは、信号線が奇数かどうか(integration)を考慮。
    #     for j in range(integration_num):
    #         line_id[i*integration_num + j] = int(lines[i][j])  # 故障信号線のIDを取得
    
    # if signal_num % integration_num != 0:  # 信号線の数が奇数の場合、ペアができていない信号線のIDをリストに追加
    #     for i in range(signal_num % integration_num):
    #         line_id[len(lines) - (signal_num % integration_num) + i] = int(lines[len(lines) - (signal_num % integration_num)][i])
# cd workspace/research2/experiment
#　実行コマンド：　python3 diagnosis.py

import numpy as np
import logging
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, model_from_json
from keras.layers import Dense, Dropout, BatchNormalization, Activation
from keras.layers import LeakyReLU
from tensorflow.keras.utils import get_custom_objects
from keras import optimizers
from keras import backend as K
from tensorflow.keras.optimizers import Adam

import matplotlib
import matplotlib.pyplot as plt
from memory_profiler import memory_usage
import time
import pathlib
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

import sys
import argparse
import os
import multiprocessing
from multiprocessing import Process

import random

#グローバル変数
cir = 's344'  # 対象回路
tp_file = cir + '.vec'  # テストパターンファイル名
part_stdic_file = cir + "stdic_bi/aout_" #縮退故障辞書ファイル名の一部
part_brdic_file = cir + "brdic_bi/aout_" #ブリッジ故障辞書ファイルの一部
diagnosis_dir = cir + "diagnosis_data/" #故障診断を行うためのデータを保存するフォルダ
target_fault_num = 30  # 故障診断対象の故障の数⇒全ての故障の中からtarget_fault_num個をランダムに選択する
correct_output_file = 'correct_output/' + cir + '_correct_output'  # 正常な回路の出力ファイル
input_data_file = cir + 'input'  # 入力データファイル
correct_data_file = cir + '分割正解データ' + '/' + cir + 'integrated_output'  # 統合後の正解データファイル
suplit_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_num'  # モデルの分割数が保存されたファイル
suplit_data_num_file = cir + '分割正解データ' + '/' + cir + 'suplit_data_num'  # データの分割数が保存されたファイル
input_data_num = None  # 1個のモデルにおける学習データ数
input_node_num = None  #入力層におけるノード数 初期値は8　学習結果によって変更
output_node_num = None #　出力層のノード数＝分割数による
num_models = None  # 学習させるモデルの数
model_folder = cir + 'sepmodel'  # 学習済みモデルを保存するフォルダ
fault_line_num = None  # 故障診断対象の回路における故障信号線の総数

processes = 8  # 並列処理のプロセス数

def mk_input_data():
    # 入力データファイルを開いてデータを読み込む
    with open(input_data_file) as f:
        lines = [_.replace(",", "").replace("\n", "") for _ in f.readlines()]

    # print(lines)
    
    global input_data_num
    input_data_num = int(len(lines))      #学習データ数を設定。学習データ数は入力データの行数
    global input_node_num               #グローバル変数を書き換え
    input_node_num = int(len(lines[0]))      #入力ノード数を設定。入力ノード数は入力データの各行の要素数

    int_lines = [list(map(int, _)) for _ in lines]  #list要素の型をint型に変換

    return np.array(int_lines)


def mk_output_data(fname):
    # 正解データファイルを開いてデータを読み込む
    with open(fname) as f:
        lines = [_.replace("\n", "") for _ in f.readlines()]
    
    lines = [value.split(",") for value in lines] # カンマ区切りで各信号線をリストに格納
    
    # print("dsadsa")
    # print(lines[0])

    for i in range(len(lines)):
        for j in range(len(lines[i])):
            lines[i][j] = float(lines[i][j])
    
    # print("gaga")
    # print(lines[0])
    
    global output_node_num               #グローバル変数を書き換え
    output_node_num = int(len(lines[0]))      #出力ノード数を設定。出力ノード数は正解データの各行の要素数


if __name__ == '__main__':
    
    # テストパターン数を取得
    with open(tp_file, 'r') as f:
        line = f.readline()
        tp_num = int(line.split()[0])  # テストパターン数
        print(tp_num)

    # 故障診断対象の回路における故障の総数を取得
    part_stdic_file0 =  part_stdic_file + str(0)  # 故障辞書ファイルの1つ
    with open(part_stdic_file0, 'r') as f:
        fault_inf = f.readline()  # 故障情報
        fault_num = int(fault_inf.split()[2])  # 対象回路で起こりうる故障数⇒この中から30個をランダムに選択する
        lines = f.readlines()  # 故障辞書ファイルの全行を読み込む
        print(fault_num)
        print(lines)
        # print(suplit_num)

    # 故障診断対象となる故障のIDをtarget_fault_num個ランダムに選択
    target_fault_id = random.sample(range(fault_num), target_fault_num)
    target_fault_id = sorted(target_fault_id) # 小さい番号順に並び替える
    with open(diagnosis_dir + 'fault_id', 'w') as f: # 故障IDをファイルに保存
        for i in range(target_fault_num - 1):
            f.write(str(target_fault_id[i]) + ',')
        f.write(str(target_fault_id[target_fault_num - 1]) + '\n')

    print(target_fault_id)

    # 診断対象故障の出力値をファイルに保存
    for i in range(tp_num):
      stdic_file = part_stdic_file + str(i)  # 故障辞書ファイルを指定
      with open(stdic_file, 'r') as f:
        fault_inf = f.readline()  # 故障情報
        lines = f.readlines()  # 故障辞書ファイルの全行を読み込む

      # 各辞書におけるIDと出力値を取得
      fault_output_value = {}  # 故障出力値を格納する辞書
      for j in range(0, len(lines), 2):
          id = int(lines[j].split()[1])
          fault_output_value[id] = lines[j+1]

      # 診断対象IDに対応する出力値だけを抽出
      fault_output_value = [fault_output_value[id] for id in target_fault_id if id in fault_output_value]
      fault_output_value = [s.replace('\n', '') for s in fault_output_value] # 改行コードを削除
      #   print(fault_output_value)

      with open(diagnosis_dir + 'fault_output' + str(i), 'w') as f:
          for j in range(len(fault_output_value)):
              f.write(fault_output_value[j] + '\n')
    
    print(fault_output_value)
    
    #故障出力を正常な回路出力と比較して、パスフェイル情報を取得する
    correct_output_value = []  # 正常な回路出力を格納するリスト
    with open(correct_output_file, 'r') as f:
        lines = f.readlines()  # 正常な回路出力の全行を読み込む
        for i in range(1, len(lines), 2):
            correct_output_value.append(lines[i].replace('\n', ''))

    print(correct_output_value)
    
    # 各故障の種類における回路出力と正常な回路出力を比較して、パスフェイル情報を取得
    fault_output_value = [[0 for _ in range(tp_num)] for _ in range(target_fault_num)]  # 故障出力値を格納する二次元リスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される
    for i in range(tp_num):
        with open(diagnosis_dir + 'fault_output' + str(i), 'r') as f:
            lines = f.readlines()  # fault_output~ファイルから故障出力値の全行を読み込む
            for j in range(target_fault_num):
                fault_output_value[j][i] = lines[j].replace('\n', '')
    
    print(fault_output_value)
    
    # 故障出力値と正常な回路出力を比較して、パスフェイル情報を取得
    for i in range(target_fault_num):
        with open(diagnosis_dir + 'pass_fail' + str(i), 'w') as f:
            for j in range(tp_num):
                if correct_output_value[j] == fault_output_value[i][j]:
                    pass_fail = '0' # pass 正常な回路出力と一致
                else:
                    pass_fail = '1' # fail 故障が発生した回路の出力値と正常な回路出力が一致しない
                
                f.write(pass_fail)

    
  # 学習済みの機械学習モデルに入力データを与えて、出力を取得する
    # 入力データをファイルから読み込む
    input_data = mk_input_data()

    # 分割モデルの数を取得
    with open(suplit_num_file, 'r') as f:
        num_models = int(f.readline())
    
    # 診断対象の回路における故障信号線の総数を取得
    with open(cir + 'output', 'r') as f:
        line = f.readline().replace(",", "").replace("\n", "")
        global faulet_line_num
        fault_line_num = int(len(line))  # 診断対象の故障信号線の総数
        # print(fault_line_num)

    model_pass_fail = [[0 for _ in range(tp_num)] for _ in range(fault_line_num)]  # 故障出力値を格納するリスト fault_output_value[30個ランダムに選んだ故障][テストパターン数]。fault_output_value[2][4]には、ランダムに選んだ故障のうち2番目に選んだ故障が発生した回路にテストパターン4を入力したときの出力値が格納される
    
    with open(cir + 'pair_list', 'r') as f:
        integration_num = int(f.readline().split()[1])  # 統合数
        # print(integration_num)
        signal_num = int(f.readline().split()[1])  # 信号線の数
        # print(signal_num)
        lines = [line.split() for line in f.readlines()]  # 故障信号線のペアを取得
        print(lines)
    
    # 信号線ペアのIDをリストに格納
    print("line_id:" + str(integration_num * len(lines)))
    line_id = [0 for _ in range(signal_num)]  # 故障信号線のIDを格納するリスト
    for i in range(len(lines) - (signal_num%integration_num)):    # signal_num%integration_numは、信号線が奇数かどうか(integration)を考慮。
        for j in range(integration_num):
            line_id[i*integration_num + j] = int(lines[i][j])  # 故障信号線のIDを取得
    
    if signal_num % integration_num != 0:  # 信号線の数が奇数の場合、ペアができていない信号線のIDをリストに追加
        for i in range(signal_num % integration_num):
            line_id[len(lines) - (signal_num % integration_num) + i] = int(lines[len(lines) - (signal_num % integration_num)][i])
    
    print(line_id)

    # データを何個ずつ分割したのかを取得
    with open(suplit_data_num_file, 'r') as f:
        suplit_data_num = int(f.readline().replace("\n", ""))  # データの分割数を取得
        print(suplit_data_num)

    for model_id in range(num_models):

        # モデルの読み込み
        with open(model_folder + '/' + cir + 'model_' + str(model_id) + '.tflite', 'rb') as f:  # モデルを読み込むファイルを開く
            tflite_model = f.read()

        # モデルの評価
        interpreter = tf.lite.Interpreter(model_content=tflite_model)  # TFLite形式のモデルを読み込む。保存されたTFLiteモデルをメモリに読み込み、推論を行う準備をするためのインタープリターを作成します。
        interpreter.allocate_tensors()  # #メモリを確保。モデルが使用するテンソル（データ構造）をメモリに割り当てます。TFLiteモデルをインタープリターにロードするだけでは、テンソルのメモリは割り当てられていません。この行を実行することで、モデルが推論に必要なメモリを確保します。

        input_details = interpreter.get_input_details()  # モデルの入力テンソルの詳細を取得。モデルの入力に関する詳細情報をリスト形式で返します。各要素は、入力テンソルの形状、データ型、名前などの情報を含む辞書です。
        output_details = interpreter.get_output_details()  # モデルの出力テンソルの詳細を取得。モデルの出力に関する詳細情報をリスト形式で返します。各要素は、出力テンソルの形状、データ型、名前などの情報を含む辞書です。
        
        for i in range(input_data_num):
            
            input_shape = input_details[0]['shape']  # 入力データの形状を取得.先ほど取得したinput_detailsのリストの中から、形状情報を取得。input_details[0]['shape']は、入力テンソルの形状を取得するためのコードです。['shape']は、辞書内の shape キーを指定しています。

            reshape_input_data = np.reshape(input_data[i], input_shape)  # NumPy配列の形状を指定した形 (input_shape) に変更します。これにより、入力データの形状がモデルの入力テンソルの形状と一致します。


            # モデルの入力データを設定
            interpreter.set_tensor(input_details[0]['index'], np.array(reshape_input_data, dtype=np.float32))   # モデルの入力テンソルにデータを設定します。入力テンソルのインデックス、データを指定します。[0]['index']は、入力テンソルのインデックスを取得するためのコードです。

            # モデルの推論を実行
            interpreter.invoke()

            # モデルの出力データを取得
            output_data = interpreter.get_tensor(output_details[0]['index'])

            correct_file = correct_data_file + str(model_id)  # 分割された正解データファイル名に変更　＝　s344分割正解データ/s344integrated_output + 番号
            mk_output_data(correct_file)  # 関数内でグローバル変数output_node_numを設定

            for j in range(output_node_num):
                if output_data[0][j] < 0.25:
                    output_data[0][j] = 0
                elif output_data[0][j] < 0.5:
                    output_data[0][j] = 0.375
                elif output_data[0][j] < 0.75:
                    output_data[0][j] = 0.625
                else:
                    output_data[0][j] = 1

            for j in range(output_node_num):
                if output_data[0][j] == 0:
                    model_pass_fail[line_id[j + suplit_data_num*model_id]][i] = 0
                    model_pass_fail[line_id[j + suplit_data_num*model_id + 1]][i] = 0
                elif output_data[0][j] == 0.375:
                    model_pass_fail[line_id[j + suplit_data_num*model_id]][i] = 1
                    model_pass_fail[line_id[j + suplit_data_num*model_id + 1]][i] = 0
                elif output_data[0][j] == 0.625:
                    model_pass_fail[line_id[j + suplit_data_num*model_id]][i] = 0
                    model_pass_fail[line_id[j + suplit_data_num*model_id + 1]][i] = 1
                else:
                    model_pass_fail[line_id[j + suplit_data_num*model_id]][i] = 1
                    model_pass_fail[line_id[j + suplit_data_num*model_id + 1]][i] = 1

        
            print(output_data)                # output_dataには、テストパターンnのときの、故障があるかどうかの情報が格納されている

        # print(model_pass_fail)


    


  
    




22
670
['id 0 Fault 1 sa 0\n', '11111111100111000010001111\n', 'id 1 Fault 1 sa 1\n', '11111111100000111110001111\n', 'id 2 Fault 2 sa 0\n', '11111111100000111110001111\n', 'id 3 Fault 2 sa 1\n', '11111111100000111110001111\n', 'id 4 Fault 3 sa 0\n', '11111111100000111110001111\n', 'id 5 Fault 3 sa 1\n', '11111111100000111110001111\n', 'id 6 Fault 4 sa 0\n', '11111111100000111110001111\n', 'id 7 Fault 4 sa 1\n', '11111111100000111110001111\n', 'id 8 Fault 5 sa 0\n', '11111111100000111110001111\n', 'id 9 Fault 5 sa 1\n', '11111111100000111110001111\n', 'id 10 Fault 6 sa 0\n', '11111111100000111110001111\n', 'id 11 Fault 6 sa 1\n', '11111111100000111110001111\n', 'id 12 Fault 7 sa 0\n', '11111111100000111110001111\n', 'id 13 Fault 7 sa 1\n', '11111111100000111110001111\n', 'id 14 Fault 8 sa 0\n', '11111111100000111110001111\n', 'id 15 Fault 8 sa 1\n', '11111111100000111110001111\n', 'id 16 Fault 9 sa 0\n', '11111111100000111110001111\n', 'id 17 Fault 9 sa 1\n', '1111111110000011111000111